🧱 Bloque 1 – Imports y Entorno Base

Este bloque carga todas las librerías necesarias para el proceso:
- pandas, numpy para manipular y transformar datos.
- sqlalchemy para conectarse a SQL Server.
- Path para manejar rutas de archivos.
- datetime, time para fechas y tiempos.
- shutil, re, unicodedata para copias, regex y normalización de texto.

Deja listo el entorno técnico para el resto del script.

In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np
from datetime import datetime
from sqlalchemy import text
import sys
import importlib
import json
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/ripley_fraude/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path.cwd().resolve().parents[2]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()
CONFIG_DIR    = (PROJECT_ROOT / "config").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "ripley_fraude", "config_ripley_fraude.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


🗄️ Bloque 2 – Configuración de Conexión a SQL Server

Este bloque define los parámetros de conexión a la base de datos:
- Especifica server, database, schema y table de destino.
- Construye el connection_string usando pyodbc.
- Crea el engine de SQLAlchemy con:
- fast_executemany=True para cargas masivas rápidas.
- pool_pre_ping=True para evitar conexiones caídas.

Es la base para todas las operaciones de escritura hacia SQL Server.

In [ ]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


📂 Bloque 3 – Configuración de Carpeta y Hojas del Excel

Este bloque define cómo se localiza y procesa el archivo de entrada:
- Configura RUTA_ARCHIVOS donde llegan los Excel del partner.
- Indica las extensiones válidas (.xlsx).
- Define las hojas preferidas (Base, base, BASE).
- Configura una hoja de respaldo vía FALLBACK_SHEET_INDEX.

Garantiza que siempre se elija el archivo correcto y la hoja adecuada.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "ripley_fraude").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx", "")  # Extensiones de archivo permitidas

PREFERRED_SHEETS = ["Base", "base", "BASE"]
FALLBACK_SHEET_INDEX = 2


🧼 Bloque 4 – Normalización Genérica de Nombres (normalize_name)

Este bloque implementa la función de normalización de encabezados:
- Quita acentos y normaliza caracteres Unicode.
- Elimina el símbolo ° y otros caracteres especiales.
- Convierte el texto a minúsculas, reemplazando espacios y símbolos por _.

Deja todos los nombres de columnas en un formato estable y predecible.

🧹 Bloque 5 – Helpers para Eliminar la Fila TOTAL Contable

Este bloque incorpora la lógica para detectar y eliminar la última fila de totales:
- _is_nullish: identifica valores nulos, vacíos o "none".
- _nullish_ratio_last_row: mide qué porcentaje de columnas (no excluidas) están vacías en la última fila.
- _round_safe: redondea numéricos de forma segura.
- drop_last_total_strict:
    - Compara la última fila con la suma del resto para columnas monetarias (uf_bruta, afecti, exento_iva, neto).
    - Revisa que el resto de columnas esté mayoritariamente vacío.
    - Si se cumplen ambas condiciones, elimina la fila TOTAL y la muestra en consola.

Evita que la fila de sumatoria contable del Excel contamine el dataset.

In [5]:
def _is_nullish(v):
    if pd.isna(v):
        return True
    if isinstance(v, str):
        s = v.strip()
        return (s == "") or (s.lower() == "none")
    return False


def _nullish_ratio_last_row(df, exclude=()):
    """
    Calcula el ratio de valores "nulos" en la última fila,
    excluyendo las columnas indicadas.
    """
    cols = [c for c in df.columns if c not in exclude]
    if not cols:
        return 1.0

    last = df[cols].iloc[-1]   # Series
    n_null = 0
    for v in last.values:
        if _is_nullish(v):
            n_null += 1

    return n_null / len(cols)


def _round_safe(x, nd):
    try:
        return round(float(x), nd)
    except Exception:
        return np.nan


def drop_last_total_strict(
    df: pd.DataFrame,
    money_cols=("uf_bruta", "afecti", "exento_iva", "neto"),
    round_decimals={"uf_bruta": 4, "afecti": 4, "exento_iva": 4, "neto": 2},
    null_check_exclude=(),
    null_ratio_threshold=0.80,
    atol=0.5,
    rtol=5e-4,
):
    """
    Elimina la última fila si:
      A) Para cada col monetaria: round(last, nd) == round(sum(prev), nd)
         O bien last ≈ sum(prev) con tolerancia (rtol, atol).
      B) En las demás columnas (excepto las excluidas), tiene
         >= null_ratio_threshold de nulos.

    Muestra la fila eliminada en consola.
    """
    if df is None or df.empty or len(df) < 2:
        print("DF vacío o con <2 filas: nada que eliminar.")
        return df

    last_idx = df.index[-1]
    prev = df.iloc[:-1]
    last = df.iloc[-1]

    # --- A) Chequeo de sumas ---
    sum_flags = {}
    details = {}
    for c in money_cols:
        if c not in df.columns:
            sum_flags[c] = False
            details[c] = "columna no existe"
            continue

        prev_sum = pd.to_numeric(prev[c], errors="coerce").sum(skipna=True)
        last_val = pd.to_numeric(pd.Series([last[c]]), errors="coerce").iloc[0]

        nd = round_decimals.get(c, 2)
        prev_sum_r = _round_safe(prev_sum, nd)
        last_val_r = _round_safe(last_val, nd)

        ok_round = (not np.isnan(prev_sum_r)) and (not np.isnan(last_val_r)) and (last_val_r == prev_sum_r)
        ok_close = (
            np.isfinite(prev_sum) and np.isfinite(last_val) and
            np.isclose(float(last_val), float(prev_sum), rtol=rtol, atol=atol)
        )
        sum_flags[c] = bool(ok_round or ok_close)
        details[c] = f"last={last_val} (r{nd}={last_val_r}) vs sum(prev)={prev_sum} (r{nd}={prev_sum_r})"

    all_sums_ok = all(sum_flags.values())

    # --- B) Chequeo de nulos ---
    df_for_nulls = df[[c for c in df.columns if c not in money_cols]]
    null_ratio = _nullish_ratio_last_row(df_for_nulls, exclude=null_check_exclude)
    nulls_ok = (null_ratio >= null_ratio_threshold)

    # --- C) Decisión ---
    if all_sums_ok and nulls_ok:
        print("🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…\n")
        removed = df.loc[[last_idx]].copy()

        print("🧾 Fila eliminada:")
        try:
            display(removed)  # Jupyter
        except NameError:
            print(removed.to_string(index=False))

        out = df.iloc[:-1].reset_index(drop=True)
        return out

    # Diagnóstico si no elimina
    print("❎ No se eliminó la última fila.")
    print(" - Sumas por columna:")
    for c in money_cols:
        if c in sum_flags:
            print(f"   {c}: match={sum_flags[c]} | {details.get(c)}")
        else:
            print(f"   {c}: columna no existe")
    print(f" - Null ratio última fila (no monetarias): {null_ratio:.2%} (umbral {null_ratio_threshold:.0%})")
    return df


🔍 Bloque 6 – Selección del Archivo Más Reciente y Hoja Objetivo

Este bloque se encarga de localizar qué archivo procesar:
- Valida que la carpeta configurada exista.
- Lista y ordena los archivos por fecha de modificación, de más reciente a más antiguo.
- Selecciona el archivo más reciente (archivo_origen).
- Intenta abrirlo y listar sus hojas.
- Selecciona la hoja según prioridad (PREFERRED_SHEETS) o usa la de respaldo.
- Si el archivo está bloqueado (OneDrive/Excel), crea y usa una copia temporal.

Define el punto de partida del proceso para cada corrida.

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen  = info["archivo_origen"]
excel_path_used = info["excel_path_used"]
_tmp_copy_path  = info["tmp_copy_path"]
target          = info["target_sheet"]
sheet_names     = info["sheet_names"]

print(f"Archivo: {archivo_origen.name}")
print(f"Hoja seleccionada: {target}")


🪪 Bloque 7 – Lectura Segura del Excel + Limpieza Inicial + Eliminación de Fila TOTAL

Este bloque construye el DataFrame crudo y aplica la limpieza temprana:
- Implementa read_excel_safe para leer la hoja objetivo con control de errores.
- Elimina la copia temporal en caso de haberla utilizado.
- Limpia valores textuales especiales ("None", "nan", "NaN").
- Normaliza los nombres de columnas con normalize_name.
- Aplica drop_last_total_strict para eliminar la posible fila TOTAL contable del final.

El resultado es un df_raw limpio y consistente para seguir procesando.

In [7]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

if _tmp_copy_path is not None and _tmp_copy_path.exists():
    try:
        _tmp_copy_path.unlink()
        print(f"🗑️ Copia temporal eliminada: {_tmp_copy_path}")
    except Exception as e:
        print(f"⚠️ No se pudo borrar la copia temporal '{_tmp_copy_path}': {e}")

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))


df_raw = drop_last_total_strict(
    df_raw,
    money_cols=("uf_bruta", "afecti", "exento_iva", "neto"),
    round_decimals={"uf_bruta": 4, "afecti": 4, "exento_iva": 4, "neto": 4},
    null_check_exclude=(),   # si después agregas columnas tipo nombre_archivo aquí, las puedes excluir
)

df_raw.head()

Encabezados normalizados: ['obs', 'fecha_cargo', 'num_cuota', 'cuota_prima', 'propuesta_com', 'rut_cliente', 'seguro', 'compania', 'valor_en_uf', 'rut_cliente', 'fecha_vcto', 'uf_bruta', 'afecto', 'exento', 'iva', 'afecti', 'exento_iva', 'neto', 'iva', 'total', 'poliza', 'unnamed_21', 'unnamed_22', 'unnamed_23', 'unnamed_24', 'unnamed_25']
❎ No se eliminó la última fila.
 - Sumas por columna:
   uf_bruta: match=False | last=0.007 (r4=0.007) vs sum(prev)=443.828 (r4=443.828)
   afecti: match=False | last=0.0054909254615024 (r4=0.0055) vs sum(prev)=362.79824176807153 (r4=362.7982)
   exento_iva: match=False | last=0.000465798700812 (r4=0.0005) vs sum(prev)=12.09809229599424 (r4=12.0981)
   neto: match=False | last=0.0059567241623145 (r4=0.006) vs sum(prev)=374.8963340640661 (r4=374.8963)
 - Null ratio última fila (no monetarias): 50.00% (umbral 80%)


,obs,fecha_cargo,num_cuota,cuota_prima,propuesta_com,rut_cliente,seguro,compania,valor_en_uf,rut_cliente,...,exento_iva,neto,iva,total,poliza,unnamed_21,unnamed_22,unnamed_23,unnamed_24,unnamed_25
0,DMV FRAUDE,2025-11-03 00:00:00,162,265,150583388,2606790-1,SEGURO FRAUDE RIPLEY III,SEGUROS GENERALES SURAMERICANA S.A.,0.0067,2606790-1,...,0.0008579981614325112,0.006019344244262334,0.0009806557557376662,0.007,6597903,362.8037326935443,,,,
1,DMV FRAUDE,2025-11-03 00:00:00,132,3062,154875437,3580786-1,SEGURO FRAUDE RIPLEY,SEGUROS GENERALES SURAMERICANA S.A.,0.0773,3580786-1,...,0.0051250463431936075,0.06552416706319898,0.01147583293680102,0.077,6597902,12.09855809469553,,,,
2,DMV FRAUDE,2025-11-03 00:00:00,109,265,158616638,4102187-K,SEGURO FRAUDE BCO Y CCSS,SEGUROS GENERALES SURAMERICANA S.A.,0.0067,4102187-K,...,0.00046579870081209105,0.005956724162314536,0.0010432758376854644,0.007,6597902,68.93270921177341,443.8350000000132,,,
3,DMV FRAUDE,2025-11-03 00:00:00,86,3062,161763345,4152402-2,SEGURO FRAUDE BCO Y CCSS,SEGUROS GENERALES SURAMERICANA S.A.,0.0773,4152402-2,...,0.005123785708933001,0.0655239657854599,0.01147603421454011,0.077,6597902,,,,,
4,DMV FRAUDE,2025-11-03 00:00:00,91,4112,161166058,4249416-K,SEGURO FRAUDE RIPLEY CB,SEGUROS GENERALES SURAMERICANA S.A.,0.1038,4249416-K,...,4.373392056913085e-05,0.08740194071000683,0.016598059289993163,0.104,6597901,,,,,


🧩 Bloque 8 – Separación de RUT/DV para Deudor y Cliente

Este bloque estandariza la estructura de los RUT:
- Usa split_rut_dv_hyphen para separar valores tipo 12.345.678-9.
- Detecta las dos columnas rut_cliente originales (una para deudor, otra para cliente).
- Genera columnas nuevas: rut_deudor, dv_deudor, rut_cliente, dv_cliente.
- Elimina las columnas duplicadas originales.

Permite trabajar con RUT y DV por separado, de manera clara y coherente.

In [8]:
def split_rut_dv_hyphen(series: pd.Series):
    """
    Acepta '12.345.678-9' o '12345678-K'
    Devuelve (RUT_sin_puntos_ni_guion, DV)
    """
    s = series.astype(str).str.upper().str.replace(r"\.", "", regex=True).str.strip()
    m = s.str.extract(r"^\s*(?P<RUT>\d+)-(?P<DV>[\dK])\s*$")
    return m["RUT"], m["DV"]

idxs_exact = [i for i, c in enumerate(df_raw.columns) if c == "RUT_CLIENTE"]
if len(idxs_exact) < 2:
    idxs_variants = [i for i, c in enumerate(df_raw.columns)
                     if re.fullmatch(r"rut_cliente(?:[_. -]\d+)?", str(c))]
else:
    idxs_variants = idxs_exact

if len(idxs_variants) < 2:
    raise ValueError(f"Se esperaban 2 columnas 'rut_cliente' y solo se detectaron {len(idxs_variants)}.")

# Primera ocurrencia -> DEUDOR ; Segunda -> CLIENTE
i_deudor, i_cliente = idxs_variants[0], idxs_variants[1]
src_deudor  = df_raw.iloc[:, i_deudor]
src_cliente = df_raw.iloc[:, i_cliente]

rut_d, dv_d = split_rut_dv_hyphen(src_deudor)
rut_c, dv_c = split_rut_dv_hyphen(src_cliente)


for col in ["rut_deudor","dv_deudor","rut_cliente","dv_cliente"]:
    if col not in df_raw.columns:
        df_raw[col] = pd.NA

df_raw = df_raw.drop(df_raw.columns[idxs_variants], axis=1)

df_raw["rut_deudor"]  = rut_d          
df_raw["dv_deudor"]   = dv_d           
df_raw["rut_cliente"] = rut_c          
df_raw["dv_cliente"]  = dv_c           


df_raw.head()

,obs,fecha_cargo,num_cuota,cuota_prima,propuesta_com,seguro,compania,valor_en_uf,fecha_vcto,uf_bruta,...,poliza,unnamed_21,unnamed_22,unnamed_23,unnamed_24,unnamed_25,rut_deudor,dv_deudor,dv_cliente,rut_cliente
0,DMV FRAUDE,2025-11-03 00:00:00,162,265,150583388,SEGURO FRAUDE RIPLEY III,SEGUROS GENERALES SURAMERICANA S.A.,0.0067,20251120,0.007,...,6597903,362.8037326935443,,,,,2606790,1,1,2606790
1,DMV FRAUDE,2025-11-03 00:00:00,132,3062,154875437,SEGURO FRAUDE RIPLEY,SEGUROS GENERALES SURAMERICANA S.A.,0.0773,20251120,0.077,...,6597902,12.09855809469553,,,,,3580786,1,1,3580786
2,DMV FRAUDE,2025-11-03 00:00:00,109,265,158616638,SEGURO FRAUDE BCO Y CCSS,SEGUROS GENERALES SURAMERICANA S.A.,0.0067,20251120,0.007,...,6597902,68.93270921177341,443.8350000000132,,,,4102187,K,K,4102187
3,DMV FRAUDE,2025-11-03 00:00:00,86,3062,161763345,SEGURO FRAUDE BCO Y CCSS,SEGUROS GENERALES SURAMERICANA S.A.,0.0773,20251120,0.077,...,6597902,,,,,,4152402,2,2,4152402
4,DMV FRAUDE,2025-11-03 00:00:00,91,4112,161166058,SEGURO FRAUDE RIPLEY CB,SEGUROS GENERALES SURAMERICANA S.A.,0.1038,20251120,0.104,...,6597901,,,,,,4249416,K,K,4249416


📦 Bloque 9 – Construcción del DataFrame Canónico (df_can)

Este bloque arma el modelo de datos intermedio:
- Usa la función pick para mapear columnas origen → destino tolerando nombres distintos.
- Extrae valores clave: fechas, cuotas, propuestas, seguros, póliza, etc.
- Construye df_can con las columnas finales que usará el modelo SQL.
- Asigna NOMBRE_ARCHIVO con el nombre original del archivo procesado.

Es el puente entre el Excel desordenado y el modelo de datos limpio.

In [9]:
def pick(df, *names, required=False, default=None):
    for n in names:
        if n in df.columns:
            col = df[n]
            if isinstance(col, pd.DataFrame):   
                return col.iloc[:, 0]           
            return col
    if required:
        raise KeyError(f"No se encontró ninguna de las columnas: {names}")
    return pd.Series([default] * len(df), index=df.index)


# Origen → Destino
fecha_cargo         = pick(df_raw, "fecha_cargo", "fechacargo", "fec_cargo", "feccargo")
num_cuota           = pick(df_raw, "num_cuota", "nro_cuota", "numerocuota")
cuota_prima         = pick(df_raw, "cuota_prima", "cuotaprima", "prima_bruta_clp", "primabrutaclp")
propuesta_com       = pick(df_raw, "propuesta_com", "propuestacom", "num_propuesta", "numpropuesta")
rut_deudor          = pick(df_raw, "rut_deudor", "rutdeudor")
dv_deudor           = pick(df_raw, "dv_deudor", "dvdeudor")
seguro              = pick(df_raw, "seguro", "nombre_seguro", "nombreseguro")
compania            = pick(df_raw, "compania", "compania_seguradora", "companiaseguradora")
prima_uf            = pick(df_raw, "valor_en_uf", "valorenuf", "prima_bruta_uf", "primabrutauf")
rut_cliente         = pick(df_raw, "rut_cliente", "rutcliente", "rut_asegurado", "rutasegurado")
dv_cliente          = pick(df_raw, "dv_cliente", "dvcliente")
fecha_vcto          = pick(df_raw, "fecha_vcto", "fechavcto", "fec_vcto", "fecvcto")
observaciones       = pick(df_raw, "observaciones", "observacion", "obs")
poliza              = pick(df_raw, "poliza", "nro_poliza", "numeropoliza")


df_can = pd.DataFrame({
    "FECHA_CARGO": fecha_cargo,
    "NUM_CUOTA": num_cuota,
    "CUOTA_PRIMA": cuota_prima,
    "PROPUESTA_COM": propuesta_com,
    "RUT_DEUDOR": rut_deudor,
    "DV_DEUDOR": dv_deudor,
    "SEGURO": seguro,
    "COMPANIA": compania,
    "PRIMA_UF": prima_uf,
    "RUT_CLIENTE": rut_cliente,
    "DV_CLIENTE": dv_cliente,
    "FECHA_VCTO": fecha_vcto,
    "OBSERVACION": observaciones,
    "POLIZA": poliza,
    "NOMBRE_ARCHIVO": archivo_origen.name,
    
})

df_can.head()

,FECHA_CARGO,NUM_CUOTA,CUOTA_PRIMA,PROPUESTA_COM,RUT_DEUDOR,DV_DEUDOR,SEGURO,COMPANIA,PRIMA_UF,RUT_CLIENTE,DV_CLIENTE,FECHA_VCTO,OBSERVACION,POLIZA,NOMBRE_ARCHIVO
0,2025-11-03 00:00:00,162,265,150583388,2606790,1,SEGURO FRAUDE RIPLEY III,SEGUROS GENERALES SURAMERICANA S.A.,0.0067,2606790,1,20251120,DMV FRAUDE,6597903,Base Fraude RIPLEY 112025 (MT1225).xlsx
1,2025-11-03 00:00:00,132,3062,154875437,3580786,1,SEGURO FRAUDE RIPLEY,SEGUROS GENERALES SURAMERICANA S.A.,0.0773,3580786,1,20251120,DMV FRAUDE,6597902,Base Fraude RIPLEY 112025 (MT1225).xlsx
2,2025-11-03 00:00:00,109,265,158616638,4102187,K,SEGURO FRAUDE BCO Y CCSS,SEGUROS GENERALES SURAMERICANA S.A.,0.0067,4102187,K,20251120,DMV FRAUDE,6597902,Base Fraude RIPLEY 112025 (MT1225).xlsx
3,2025-11-03 00:00:00,86,3062,161763345,4152402,2,SEGURO FRAUDE BCO Y CCSS,SEGUROS GENERALES SURAMERICANA S.A.,0.0773,4152402,2,20251120,DMV FRAUDE,6597902,Base Fraude RIPLEY 112025 (MT1225).xlsx
4,2025-11-03 00:00:00,91,4112,161166058,4249416,K,SEGURO FRAUDE RIPLEY CB,SEGUROS GENERALES SURAMERICANA S.A.,0.1038,4249416,K,20251120,DMV FRAUDE,6597901,Base Fraude RIPLEY 112025 (MT1225).xlsx


📅 Bloque 10 – Nombre Canónico de Archivo según Período

Este bloque estandariza el identificador de archivo:
- Define el diccionario MESES con nombres y abreviaturas (ENE, FEB, SEPT, etc.).
- Implementa _extract_yyyymm_from_name para detectar YYYYMM desde el nombre del archivo, incluso con abreviaturas y puntos.
- Construye un nombre canónico del tipo: Base Fraude RIPLEY YYYYMM.xlsx
- Si no puede extraer el período, genera un nombre genérico con timestamp.
- Actualiza df_can["NOMBRE_ARCHIVO"] con el nombre canónico.

Uniforma el identificador del archivo para trazabilidad y control de versiones.

In [10]:
MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09", "SEPT": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}

def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = _strip_accents(stem).upper()


    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"


    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"


    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"


    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        for mes_txt, mm in MESES.items():
            if re.search(rf"(?<![A-Z0-9]){mes_txt}[\.\s]?(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")

def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve un nombre canónico estandarizado: 'SC_MARSHYYYYMM.xlsx'
    Si no logra extraer el período (YYYYMM), devuelve un nombre genérico con aviso.
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"Base Fraude RIPLEY {yyyymm}.xlsx"
    except ValueError as e:
        print(f"⚠️ No se pudo extraer período desde el nombre '{nombre}'. "
              f"Se usará un nombre genérico. Detalle: {e}")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"Base Fraude RIPLEY{timestamp}.xlsx"



nombre_origen = archivo_origen.name
nombre_canonico = canonicalizar_planes(nombre_origen)

print("Nombre original :", nombre_origen)
print("Nombre canónico :", nombre_canonico)

df_can["NOMBRE_ARCHIVO"] = nombre_canonico

Nombre original : Base Fraude RIPLEY 112025 (MT1225).xlsx
Nombre canónico : Base Fraude RIPLEY 202511.xlsx


🗓️ Bloque 11 – Derivación del Campo ANO_MES_CARGA

Este bloque crea el período de carga a partir del nombre de archivo:
- Usa extraer_yyyymm_de_nombre_archivo sobre la columna NOMBRE_ARCHIVO.
- Extrae YYYYMM y lo almacena en ANO_MES_CARGA.
- Notifica si algún registro no pudo extraer correctamente su período.

Deja listo un campo clave para filtrado, reporting y particionamiento por mes de carga.

In [11]:
def extraer_yyyymm_de_nombre_archivo(df: pd.DataFrame, nombre_columna: str = "NOMBRE_ARCHIVO") -> pd.Series:
    return df[nombre_columna].str.extract(r"(20\d{2})(0[1-9]|1[0-2])", expand=False).apply(lambda x: ''.join(x), axis=1)


df_can["ANO_MES_CARGA"] = extraer_yyyymm_de_nombre_archivo(df_can)


if df_can["ANO_MES_CARGA"].isnull().any():
    print("⚠️ Algunos valores de ANO_MES_CARGA no pudieron ser extraídos.")
    

df_can[["NOMBRE_ARCHIVO", "ANO_MES_CARGA"]].head(3)

,NOMBRE_ARCHIVO,ANO_MES_CARGA
0,Base Fraude RIPLEY 202511.xlsx,202511
1,Base Fraude RIPLEY 202511.xlsx,202511
2,Base Fraude RIPLEY 202511.xlsx,202511


💰 Bloque 12 – Cálculo de PRIMA_NETA_UF

Este bloque enriquece el dataset con un indicador actuarial:
- Verifica que la columna PRIMA_UF exista en df_can.
- Convierte PRIMA_UF a numérico de forma segura.
- Calcula PRIMA_NETA_UF = PRIMA_UF / 1.19.
- Muestra una vista rápida de ambas columnas para verificar la lógica.

Agrega una métrica derivada relevante para análisis de negocio.

In [12]:
if "PRIMA_UF" not in df_can.columns:
    raise ValueError("No se encontró la columna 'PRIMA_UF'.")

df_can["PRIMA_UF"] = pd.to_numeric(df_can["PRIMA_UF"], errors="coerce")

df_can["PRIMA_NETA_UF"] = df_can["PRIMA_UF"] / 1.19

print("✅ Columnas creadas correctamente:")
print(df_can[["PRIMA_UF", "PRIMA_NETA_UF"]].head())

✅ Columnas creadas correctamente:
   PRIMA_UF  PRIMA_NETA_UF
0    0.0067       0.005630
1    0.0773       0.064958
2    0.0067       0.005630
3    0.0773       0.064958
4    0.1038       0.087227


⏱️ Bloque 13 – Normalización de FECHA_CARGO a Formato YYYYMMDD

Este bloque prepara la fecha de cargo para SQL:
- Valida que FECHA_CARGO exista.
- Convierte los valores a datetime usando el formato esperado.
- Luego la convierte a string YYYYMMDD, lista para ser interpretada como entero.

Alinea el formato de fecha al estándar utilizado en el modelo de SQL Server.

In [13]:
if "FECHA_CARGO" not in df_can.columns:
    raise ValueError("No se encontró la columna 'FECHA_CARGO' en el DataFrame.")


df_can["FECHA_CARGO"] = pd.to_datetime(
    df_can["FECHA_CARGO"],
    errors="coerce", 
    format="%Y-%m-%d %H:%M:%S"
)

df_can["FECHA_CARGO"] = df_can["FECHA_CARGO"].dt.strftime("%Y%m%d")

print("✅ FECHA_CARGO transformada correctamente:")
print(df_can["FECHA_CARGO"].head())

✅ FECHA_CARGO transformada correctamente:
0    20251103
1    20251103
2    20251103
3    20251103
4    20251103
Name: FECHA_CARGO, dtype: object


🧪 Bloque 14 – Tipificación Final y Construcción de df_sql

Este bloque deja los tipos de datos listos para la carga:
- Define columnas INT_COLS, FLOAT_COLS, BIGINT_COLS, STR_COLS.
- Implementa _to_num_series para convertir y limpiar numéricos.
- Convierte:
    - BIGINT_COLS y INT_COLS a Int64 (enteros que aceptan nulos), con aviso y redondeo si aparecen decimales.
    - FLOAT_COLS a float64.
- Limpia y recorta las columnas de texto según longitud objetivo.
- Imprime los dtypes finales y revisa nulos en columnas críticas.
- Construye df_sql solo con las columnas esperadas y en el orden correcto.

Deja un DataFrame perfectamente alineado al esquema de la tabla Ripley_Fraude.

In [14]:
INT_COLS    = ["FECHA_CARGO","NUM_CUOTA","RUT_DEUDOR", "RUT_CLIENTE", "FECHA_VCTO", "ANO_MES_CARGA"]
FLOAT_COLS  = ["CUOTA_PRIMA", "PRIMA_UF", "PRIMA_NETA_UF"]
BIGINT_COLS = ["PROPUESTA_COM", "POLIZA"]
STR_COLS    = ["DV_DEUDOR", "SEGURO", "COMPANIA", "DV_CLIENTE", "OBSERVACION", "CAMPAÑAS", "NOMBRE_ARCHIVO"]
DATE_COLS   = []

def _to_num_series(s: pd.Series) -> pd.Series:
    s2 = s.astype(str).str.strip().replace({
        "": np.nan,
        "None": np.nan,
        "none": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
    })
    return pd.to_numeric(s2, errors="coerce")


def convert_to_datetime(df: pd.DataFrame, date_columns: list) -> pd.DataFrame:
    """
    Convierte las columnas de fechas de formato object a formato datetime (YYYY-MM-DD).
    Si la conversión falla, reemplaza con NaT (Not a Time).
    """
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
            # Reemplaza cualquier valor inválido con NaT (not a time) si no se puede convertir
            # Esto asegura que las fechas no válidas sean manejadas adecuadamente.
    return df

df_can = convert_to_datetime(df_can, DATE_COLS)


# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        s = _to_num_series(df_can[c])

        frac_mask = (s.notna()) & (s % 1 != 0)
        if frac_mask.any():
            print(f"⚠️ Columna '{c}' (INT) tiene {frac_mask.sum()} valores con decimales. Se redondearán antes de convertir a Int64.")
            # Aproximar al entero más cercano
            s = s.round(0)

        df_can[c] = s.astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")



if "DV_DEUDOR" in df_can.columns:
    df_can["DV_DEUDOR"] = df_can["DV_DEUDOR"].astype("string").str.strip().str.slice(0, 1)

if "SEGURO" in df_can.columns:
    df_can["SEGURO"] = df_can["SEGURO"].astype("string").str.strip().str.slice(0, 255)

if "COMPANIA" in df_can.columns:
    df_can["COMPANIA"] = df_can["COMPANIA"].astype("string").str.strip().str.slice(0, 255)

if "DV_CLIENTE" in df_can.columns:
    df_can["DV_CLIENTE"] = df_can["DV_CLIENTE"].astype("string").str.strip().str.slice(0, 1)

if "OBSERVACION" in df_can.columns:
    df_can["OBSERVACION"] = df_can["OBSERVACION"].astype("string").str.strip().str.slice(0, 255)

if "CAMPAÑAS" in df_can.columns:
    df_can["CAMPAÑAS"] = df_can["CAMPAÑAS"].astype("string").str.strip().str.slice(0, 255)

if "NOMBRE_ARCHIVO" in df_can.columns:
    df_can["NOMBRE_ARCHIVO"] = df_can["NOMBRE_ARCHIVO"].astype("string").str.strip().str.slice(0, 255)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

criticas = ["RUT_DEUDOR" "PRIMA_UF", "POLIZA", "NOMBRE_ARCHIVO"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "FECHA_CARGO","NUM_CUOTA", "CUOTA_PRIMA", "PROPUESTA_COM", "RUT_DEUDOR", "DV_DEUDOR", "SEGURO", "COMPANIA", "PRIMA_UF", "RUT_CLIENTE", 
    "DV_CLIENTE", "FECHA_VCTO", "OBSERVACION", "POLIZA", "CAMPAÑAS", "NOMBRE_ARCHIVO", "ANO_MES_CARGA", "PRIMA_NETA_UF",
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

✅ dtypes DESPUÉS:
 FECHA_CARGO                Int64
NUM_CUOTA                  Int64
CUOTA_PRIMA              float64
PROPUESTA_COM              Int64
RUT_DEUDOR                 Int64
DV_DEUDOR         string[python]
SEGURO            string[python]
COMPANIA          string[python]
PRIMA_UF                 float64
RUT_CLIENTE                Int64
DV_CLIENTE        string[python]
FECHA_VCTO                 Int64
OBSERVACION       string[python]
POLIZA                     Int64
NOMBRE_ARCHIVO    string[python]
ANO_MES_CARGA              Int64
PRIMA_NETA_UF            float64
dtype: object

🔎 Nulos en columnas críticas:
 - POLIZA: 0 nulos
 - NOMBRE_ARCHIVO: 0 nulos

📦 df_sql listo con columnas: ['FECHA_CARGO', 'NUM_CUOTA', 'CUOTA_PRIMA', 'PROPUESTA_COM', 'RUT_DEUDOR', 'DV_DEUDOR', 'SEGURO', 'COMPANIA', 'PRIMA_UF', 'RUT_CLIENTE', 'DV_CLIENTE', 'FECHA_VCTO', 'OBSERVACION', 'POLIZA', 'NOMBRE_ARCHIVO', 'ANO_MES_CARGA', 'PRIMA_NETA_UF']


🧾 Bloque 15 – Validación y Resumen de NOMBRE_ARCHIVO en df_sql

Este bloque controla qué archivos están presentes en la carga:
- Verifica que NOMBRE_ARCHIVO exista en df_sql.
- Normaliza el contenido como string limpio.
- Obtiene los distintos NOMBRE_ARCHIVO existentes (vals).
- Lanza error si no hay ningún valor válido.
- Muestra cuántas filas aporta cada archivo.

Es un punto de control antes de tocar la base de datos.

In [15]:
assert "NOMBRE_ARCHIVO" in df_sql.columns, "Falta la columna 'NOMBRE_ARCHIVO' en df_sql."

df_sql["NOMBRE_ARCHIVO"] = (
    df_sql["NOMBRE_ARCHIVO"]
    .astype("string")
    .str.strip()
)


vals = [
    v for v in df_sql["NOMBRE_ARCHIVO"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'NOMBRE_ARCHIVO'.")

counts = (
    df_sql["NOMBRE_ARCHIVO"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} NOMBRE_ARCHIVO distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 NOMBRE_ARCHIVO distintos en el df_sql:
   - Base Fraude RIPLEY 202511.xlsx: 8726 filas


🚀 Bloque 16 – Carga Controlada en SQL Server + Reemplazo por Archivo

Este bloque implementa la carga final al Data Warehouse:
- Abre una transacción única con engine.begin().
- Para cada NOMBRE_ARCHIVO en vals:
- Consulta cuántas filas existen ya en {schema}.{table} para ese archivo.
- Si hay datos previos:
    - Los elimina (reemplazo controlado por archivo).
- Inserta las nuevas filas desde df_sql usando to_sql.
- Construye y muestra un resumen de:
    - Archivos reemplazados vs archivos nuevos.
    - Cantidad de filas cargadas por archivo.

Implementa un flujo de upsert por archivo robusto y deja la carpeta de entrada limpia tras cada ejecución.

In [16]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    for nombre_archivo in vals: 
        df_sub = df_sql[df_sql["NOMBRE_ARCHIVO"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para NOMBRE_ARCHIVO = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para NOMBRE_ARCHIVO = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para NOMBRE_ARCHIVO = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para NOMBRE_ARCHIVO = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por NOMBRE_ARCHIVO:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")


🆕 No hay data previa para NOMBRE_ARCHIVO = 'Base Fraude RIPLEY 202511.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 8726 filas para NOMBRE_ARCHIVO = 'Base Fraude RIPLEY 202511.xlsx'.

📊 Resumen de carga por NOMBRE_ARCHIVO:
   - Base Fraude RIPLEY 202511.xlsx: 8726 filas cargadas (archivo nuevo).


🗑️ Bloque 17 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [ ]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")